# Smoke Test — Issue #11: Wire Frontend to Backend

Covers:
- `loadExample` → `POST /example` with domain, returns camelCase `ExampleResult`
- `analyzeExample` → `POST /analyze` with `example_id`, returns full `AnalyzeResponse`
- `DiagnosticCard` renders forensics data from a real backend response
- Stub removal: `retrieval_score_distribution` and `confidence_calibration` no longer in the response
- All 28 frontend + 16 backend tests green

**Prerequisites:** backend running on `http://localhost:8000` with `ANTHROPIC_API_KEY` set.
```bash
# Terminal 1 — from backend/
poetry run uvicorn main:app --reload

# Terminal 2 — from frontend/
npm run dev
```

## Cell 1 — Backend liveness check

In [ ]:
import requests

BASE_URL = "http://localhost:8000"
r = requests.get(f"{BASE_URL}/")
print(r.status_code, r.json())

**Expected output:**
```
200 {'status': 'ok'}
```

## Cell 2 — `POST /example` (mirrors `loadExample` in `lib/api.ts`)

In [ ]:
r = requests.post(f"{BASE_URL}/example", json={"domain": "techqa"})
print("Status:", r.status_code)
data = r.json()

# frontend maps these snake_case keys → camelCase
print("example_id (→ exampleId):", data["example_id"])
print("question:", data["question"][:80])
print("context_preview (→ context):", data["context_preview"][:120], "...")

**Expected output (values vary by ChromaDB sample):**
```
Status: 200
example_id (→ exampleId): techqa_...
question: <some question string>
context_preview (→ context): <first 120 chars of context> ...
```

## Cell 3 — `POST /analyze` response shape (mirrors `analyzeExample` in `lib/api.ts`)

In [ ]:
example_id = data["example_id"]
r2 = requests.post(f"{BASE_URL}/analyze", json={"example_id": example_id})
print("Status:", r2.status_code)
body = r2.json()

EXPECTED_KEYS = {
    "question", "generated_answer", "retrieved_chunks",
    "ragas", "hedging_mismatch", "chunk_attribution",
    "retrieval_distribution", "embedding_space", "query_corpus_fit",
    "recommendation", "rule_id",
}
REMOVED_KEYS = {"retrieval_score_distribution", "confidence_calibration"}

actual_keys = set(body.keys())
missing = EXPECTED_KEYS - actual_keys
leaked = REMOVED_KEYS & actual_keys

print(f"Top-level keys ({len(actual_keys)}): {sorted(actual_keys)}")
print("Missing expected keys:", missing or "none")
print("Leaked stub keys:", leaked or "none ✓")

**Expected output:**
```
Status: 200
Top-level keys (11): ['chunk_attribution', 'embedding_space', 'generated_answer',
                      'hedging_mismatch', 'query_corpus_fit', 'question',
                      'recommendation', 'retrieval_distribution', 'retrieved_chunks',
                      'ragas', 'rule_id']
Missing expected keys: none
Leaked stub keys: none ✓
```

## Cell 4 — Verdict and forensics fields

In [ ]:
print("rule_id:", body["rule_id"])
print("recommendation:", body["recommendation"])
print()

# RAGAS baseline
ragas = body["ragas"]
print(f"RAGAS — retrieval_relevance: {ragas['retrieval_relevance_score']:.2f}  faithfulness: {ragas['faithfulness_score']:.2f}")

# Hedging
hm = body["hedging_mismatch"]
print(f"Hedging — overconfident: {hm['overconfident_fraction']:.0%}  underconfident: {hm['underconfident_fraction']:.0%}  claims: {hm['total_claims']}")

# Chunk attribution
ca = body["chunk_attribution"]
print(f"Attribution — unattributed: {ca['unattributed_fraction']:.0%}  mean_score: {ca['mean_attribution_score']:.2f}  sentences: {len(ca['attribution_map'])}")

# Query-corpus fit
qcf = body["query_corpus_fit"]
print(f"Query-corpus fit — triggered: {qcf['triggered']}  mismatch_type: {qcf['mismatch_type']}")

**Expected output (values vary by sample and LLM):**
```
rule_id: R07  (or R01–R09 depending on the sample)
recommendation: Pipeline looks healthy...  (or specific issue description)

RAGAS — retrieval_relevance: 0.xx  faithfulness: 0.xx
Hedging — overconfident: x%  underconfident: x%  claims: N
Attribution — unattributed: x%  mean_score: 0.xx  sentences: N
Query-corpus fit — triggered: False  mismatch_type: None
```

## Cell 5 — End-to-end across all three domains

In [ ]:
domains = ["techqa", "finqa", "covidqa"]
rows = []
for domain in domains:
    ex = requests.post(f"{BASE_URL}/example", json={"domain": domain}).json()
    r3 = requests.post(f"{BASE_URL}/analyze", json={"example_id": ex["example_id"]})
    b = r3.json()
    rows.append({
        "domain": domain,
        "status": r3.status_code,
        "rule_id": b.get("rule_id"),
        "recommendation": b.get("recommendation", "")[:60],
        "retrieval_score_distribution_absent": "retrieval_score_distribution" not in b,
        "confidence_calibration_absent": "confidence_calibration" not in b,
    })

print(f"{'Domain':<10} {'Status':<8} {'rule_id':<8} {'Stubs absent':<14} Recommendation")
print("-" * 80)
for row in rows:
    stubs_ok = "✓" if row["retrieval_score_distribution_absent"] and row["confidence_calibration_absent"] else "✗"
    print(f"{row['domain']:<10} {row['status']:<8} {row['rule_id']:<8} {stubs_ok:<14} {row['recommendation']}")

**Expected output:**
```
Domain     Status   rule_id  Stubs absent   Recommendation
--------------------------------------------------------------------------------
techqa     200      R0x      ✓              <recommendation text>
finqa      200      R0x      ✓              <recommendation text>
covidqa    200      R0x      ✓              <recommendation text>
```
- All three domains return HTTP 200
- All three confirm `retrieval_score_distribution` and `confidence_calibration` are absent (✓)

## Cell 6 — Test suites (backend + frontend)

In [ ]:
import subprocess, os

repo = os.path.abspath("..")

backend_result = subprocess.run(
    ["poetry", "run", "pytest", "tests/test_analyze.py", "-q"],
    cwd=os.path.join(repo, "backend"),
    capture_output=True, text=True,
)
print("=== Backend ===")
print(backend_result.stdout[-800:])

frontend_result = subprocess.run(
    ["npm", "test", "--", "--passWithNoTests"],
    cwd=os.path.join(repo, "frontend"),
    capture_output=True, text=True,
)
print("=== Frontend ===")
print(frontend_result.stdout[-600:])

**Expected output:**
```
=== Backend ===
16 passed, 1 warning in X.XXs

=== Frontend ===
Test Suites: 3 passed, 3 total
Tests:       28 passed, 28 total
```